# Week 3: Evaluate honestly

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GDG-TUM/AI-ML-Track-GDG-on-Campus-TUM/blob/main/weekly-sessions/week-03-evaluate-honestly/evaluate_honestly.ipynb)

**GDG on Campus TUM · AI/ML Track**

Anyone can build a model that scores 99% on a slide. The real skill is knowing whether that number deserves to be believed. Three classic traps today:

1. **Overfitting:** memorising instead of learning
2. **Data leakage:** accidentally cheating
3. **The accuracy trap:** a "95% accurate" model that is useless

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_classification, make_moons
from sklearn.dummy import DummyClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import cross_val_score, train_test_split, validation_curve
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 0

## Trap 1: Overfitting

We make a noisy two-moons dataset and train decision trees of increasing **depth**. A deeper tree can ask more questions, so it can memorise more.

In [ ]:
X, y = make_moons(n_samples=300, noise=0.35, random_state=RANDOM_STATE)

depths = np.arange(1, 16)
train_scores, val_scores = validation_curve(
    DecisionTreeClassifier(random_state=RANDOM_STATE),
    X,
    y,
    param_name="max_depth",
    param_range=depths,
    cv=5,
)

plt.figure(figsize=(7, 4))
plt.plot(depths, train_scores.mean(axis=1), "o-", color="#EA4335", label="training score")
plt.plot(depths, val_scores.mean(axis=1), "o-", color="#4285F4", label="validation score")
plt.xlabel("tree depth (model complexity)")
plt.ylabel("accuracy")
plt.title("Training keeps climbing. Validation does not.")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

best_depth = depths[val_scores.mean(axis=1).argmax()]
print("depth with the best validation score:", best_depth)

Read the chart left to right. At first, more depth helps both lines (the model is **underfitting**: too simple). Then training accuracy heads towards 100% while validation accuracy stalls or drops (**overfitting**). The sweet spot is where validation peaks.

The fix is a habit: **choose models by validation score, never by training score.**

### Cross-validation: a fairer exam

One train/test split can be lucky or unlucky. **k-fold cross-validation** splits the data k times, trains k models, and averages. You also get a spread, which tells you how much to trust the average.

In [ ]:
tree = DecisionTreeClassifier(max_depth=int(best_depth), random_state=RANDOM_STATE)
scores = cross_val_score(tree, X, y, cv=5)
print("5 scores :", scores.round(3))
print(f"mean ± std: {scores.mean():.3f} ± {scores.std():.3f}")

## Trap 2: Data leakage (accidentally cheating)

This is the most common way beginners (and professionals!) fool themselves. Here is a famous demonstration.

We create **pure noise**: 100 people, 5,000 random measurements, and labels that are a coin flip. There is *nothing* to learn, so an honest model should score about **50%**.

The leaky workflow does what looks sensible: first keep the 20 "most useful" features, then cross-validate. The honest workflow puts feature selection *inside* each fold.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
X_noise = rng.normal(size=(100, 5000))
y_noise = rng.integers(0, 2, size=100)  # coin flips: no real signal at all

# ❌ Leaky: choose features using ALL the labels, then cross-validate
X_selected = SelectKBest(f_classif, k=20).fit_transform(X_noise, y_noise)
leaky = cross_val_score(LogisticRegression(max_iter=1000), X_selected, y_noise, cv=5).mean()

# ✅ Honest: feature selection happens inside each training fold only
honest_model = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=1000))
honest = cross_val_score(honest_model, X_noise, y_noise, cv=5).mean()

print(f"leaky  workflow accuracy: {leaky:.2f}   <- looks like a discovery. It is a mirage.")
print(f"honest workflow accuracy: {honest:.2f}   <- about 0.5, the truth about coin flips")
assert leaky > honest + 0.15

The leaky workflow "peeked" at the labels of the data it was later tested on, so it found features that happened to match *those* coin flips. Remember the rule:

> **Anything that learns from data (scaling, feature selection, imputing, encoding) must be fitted on training data only.** A scikit-learn `Pipeline` does this for you. Use one.

## Trap 3: The accuracy trap

Imagine a model that screens for a rare problem that affects 5% of cases (a fraud, a disease, a failing machine). A model that always says "no problem" is **95% accurate** and completely useless.

In [ ]:
X_imb, y_imb = make_classification(
    n_samples=2000,
    n_features=10,
    n_informative=5,
    weights=[0.95, 0.05],
    flip_y=0.01,
    random_state=RANDOM_STATE,
)
X_tr, X_te, y_tr, y_te = train_test_split(
    X_imb, y_imb, test_size=0.3, stratify=y_imb, random_state=RANDOM_STATE
)

lazy = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)
print(f"'Always say no' accuracy: {accuracy_score(y_te, lazy.predict(X_te)):.3f}")
print()
print(classification_report(y_te, lazy.predict(X_te), zero_division=0, digits=3))

Look at the row for class `1` (the rare case): **recall = 0.** The lazy model catches *none* of the cases we actually care about, yet its accuracy looks great.

* **Precision:** when the model raises an alarm, how often is it right?
* **Recall:** of all the real cases, how many did the model catch?
* **F1:** one number that balances the two.

Now a real model:

In [ ]:
real = LogisticRegression(max_iter=1000, class_weight="balanced").fit(X_tr, y_tr)
print(classification_report(y_te, real.predict(X_te), digits=3))

Notice what happened: overall **accuracy went down** (to roughly 0.85) but **recall for class 1 went from 0 to well above zero**. The model now actually finds some of the rare cases, at the price of some false alarms. Accuracy fell and the model became far more useful. Your exact numbers may differ slightly.

Which metric matters depends on the cost of each mistake. Missing a sick patient is worse than a false alarm; flagging an innocent customer as a fraudster has its own cost. **Ask the people affected, not just the data.** We return to this in Week 8 (Responsible AI).

## A checklist to keep

- [ ] Did I compare against a **baseline**?
- [ ] Did I choose settings with **validation** data, not the test set?
- [ ] Is every preprocessing step inside a **Pipeline** (no leakage)?
- [ ] Did I look at **more than accuracy** (precision, recall, the confusion matrix)?
- [ ] Did I **look at real mistakes** with my own eyes?
- [ ] Can I explain who is hurt if the model is wrong?

## 🏁 Take-home challenge

1. Make your **first Kaggle submission**: join the [Titanic competition](https://www.kaggle.com/competitions/titanic), use a `Pipeline`, and report your cross-validation score *and* your leaderboard score. Are they close? If not, why?
2. In the leakage demo, change `k=20` to `k=5` and to `k=200`. What happens to the leaky score?
3. Write three sentences about a model in real life (health, loans, exams, hiring) where accuracy alone would mislead.